In [ ]:
!pip install ipympl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from functools import partial
from scipy.fft import fft, fftshift, fftfreq, rfft

In [ ]:
"""
This cell is a cousin to the animation from Exercise 1, but without the phasor.
You ARE required to understand it, as you may be modifying it later.
Read and fill in the simulation code (next cell) first, to better understand this one.
"""
def animate_1D(ks, xs, wave_list, omegas): # call this function to make your animation
    def update(frame, lines): # an inner function for the purpose of preparing the next frame
        # wave_list contains a dictionary of 
        if frame==0:
            return
        _totals = {
            field:np.sum(
                wave_list(field)*np.exp(1j*ks*xs-1j*omegas*frame*delta_t) # propagate each wave, in each field, and add them together
            ) for field in wave_list.keys() # sums up actual contribution each each wave to each field at (x,t)
        }
        for field in wave_list.keys():
            lines[field].set_ydata(np.real(_totals[field])) # change the y-data on our plot for each plotting element
        return (*lines.values,) # change and return these elements to the main animation function


    fig, axes = plt.subplots(2) # can change no. of subplots
    for ax in axes:
        ax.set_xlabel(r"$x$")
    _totals = {key:np.sum(wave_list(key)*np.exp(1j*ks*xs)) for key in wave_list.keys()}
    lines = {}
    lines["u"] = axes[0].plot(xs, np.real(_totals["u"]))[0] # We have extracted the plotting element encoding our line plot
    lines["eta"] = axes[1].plot(xs, np.real(_totals["eta"]))[0] # and here we have a second such element
    plt.tight_layout()
    anim = animation.FuncAnimation(fig=fig, func=partial(update,lines=lines), frames=40, interval=150)
    return anim

In [ ]:
# This cell is for you to complete. It allows you to evolve a 2-field wave, once you build fns for the Fourier transfrom, 
# the dispersion relation, and the polarisation relations.

L = 10 # length of the domain
xs = np.arange(-L/2,L/2, L/256) # x in [0,L), 256 steps
t_stop = 100
delta_t = 0.1
ts = np.arange(0, t_stop, delta_t)
g = 9.8
H = 10

u = np.zeroes_like(xs) # TODO change this to reflect initial u
eta = np.zeroes_like(xs) # TODO initial eta

ks = # TODO find all ks, either using (r)fft.freq or coding up your own fn for slow fft
_u =  # TODO take the fourier transform of u. Make sure _u[i] corresponds to ks[i]
_eta = # TODO same

omegas = [] # we will append to this list below with omegas for each wave mode
# eventually will have twice the length of ks because of the 2 families of wave

wave_list = {"u": [], "eta": []} 
# a dictionary of 2 lists, each containing Fourier coeffs for each wave. Coeffs ar the same index
# in both lists correspond to the same wave

def polarisation(k, _u, _eta):
    # TODO code up a function taking _u and _eta for some k
    #      and splitting each into their forward and backward parts
    return {"u": [_u_forward, _u_backward], "eta": [_eta_forward, _eta_backward]}

def calculate_omegas(k):
    # TODO for each k, calculate the two omegas
    return [forward_omega, backward_omega]



for i, k in enumerate(ks): # for each k
    k_waves =   polarisation(k, _u[i], _eta[i])
    for key in wave_list.keys():
        wave_list[key] += k_waves[key] # appending, for each field, a list of length 2 (forward, backward)
                                       # to the existing lists
    
    omegas += calculate_omegas(k)      # appending a list of length 2 (forward, backward)

display(animate_1D(ks, xs, wave_list, omegas)) # makes and shows the animation
plt.close()

In [ ]:
# 2D animation code

def animate_2D(ks,ls, xs, ys, wave_list, omegas):
    def update_2D(frame, wave_list, plots):
        if frame==0:
            return
        
        _totals = {
            field:np.sum(
                wave_list(field)*np.exp(1j*(ks*xs+ls*ys-omegas*frame*delta_t)) # propagate each wave, in each field, and add them together
            ) for field in wave_list.keys()
        }
        for field in wave_list.keys():
            plots[field].set_array(np.real(_totals[field]))
        return (*plots.values,)
    fig, axes = plt.subplots(3) # can change no. of subplots
    for ax in axes:
        ax.set(xlabel=r"$x$", ylabel=r"$y$")
    _totals = {key:np.sum(wave_list(key)*np.exp(1j*(ks*xs+ls*ys))) for key in wave_list.keys()}
    plots = {}
    plots["u"] = axes[0].pcolor(xs, np.real(_totals["u"])) # optionally set `vmin` and `vmax` as colorbar limits for whole simulation
    plots["v"] = axes[1].pcolor(xs, np.real(_totals["v"]))
    plots["eta"] = axes[2].plot(xs, np.real(_totals["eta"]))
    plt.tight_layout()
    anim = animation.FuncAnimation(fig=fig, func=partial(update_2D, plots=plots), frames=40, interval=150)
    return anim

## Exercise 4: polarisation


#### 4.1 
Above is code to run a 1D simulation of shallow-water gravity waves, but it is incomplete. Add in the code for the polarisation relation (i.e., correctly project $u$ and $\eta$ onto $\hat u_\pm$ and $\hat\eta_\pm$) to get an animation containing both families of wave. Consider some interesting arbitrary conditions of $u$ and $\eta$:

(a)  First, show that choosing them to match either branch of the polarisation relation in 1D results in appropriately propagating (purely leftward or rightward waves). *Explain physically why the two configurations propagate the way they do, by drawing them at different points in time during their periods.* 

(b) Show that choosing either field to be zero results in two wave trains heading in opposite directions.

(c) Show that short and long waves are differently expressed in the two fields.

#### 4.2
(a) Generate plane waves--2D waves that are cousins of sinusoids given by $\cos(kx+ly+\phi)$--to understand how their structure changes as you vary $k$ and $l.$ We shall adopt the conventions $\vec\kappa = [k~l]^T$ and $\kappa=\sqrt{k^2+l^2}$


(b) Generate some 2D waveforms in physical space--Gaussians, wavepackets, square waves, whatever--, visualised as coloured plots. There's no time-evolution or propagation in this question. Use `rfft2` to transform them, and note that the Hermitian property in multiple dimensions becomes: $$\mathcal A(k,l) = A(k,l)+iB(k,l) = \mathcal A^*(-k,-l) = A(-k,-l)-iB(-k,-l)$$
so that `rfft2` halves the wavenumbers on only *one axis.* You can visualise just the absolute value of the transform to understand the energy distribution across wavenumbers, but this loses phase information.


(c) Verify that $$\vec c_p = \frac\omega{\kappa^2}\vec\kappa$$for any plane wave. Also derive an expression for $\vec c_g$ for a sharply peaked 2D wavepacket.


#### 4.3

(a) Derive the polarisation for nonrotating shallow water waves in 2D: 
$$\partial_t u + g\partial_x\eta = 0$$
$$\partial_t v + g\partial_y\eta = 0$$
$$\partial_t \eta + H(\partial_x u + \partial_y v) = 0$$
using a 2D Fourier ansatz, where each Fourier component is multiplied by $e^{i(kx+ly-\omega t)}.$ 
Note that we get 3 variables (for the three fields) but only 2 of them are independent, so we still have
only two families of wave; that propagating parallel to the wavevector $\vec\kappa,$ and that propagating antiparallel to it.

(b) Change our simulation code to carry out the Fourier transform and use the polarisation relation, but in 2D. Use the `2D_animate` function to animate their propagation. 

(c) Add 2 more fields to the simulation and the output: the vorticity $$\zeta=\partial_xv-\partial_yu,$$ and the divergence $$\delta=\partial_xu+\partial_yv.$$ Hint: these fields can be easily calculated for each plane wave component, as they are algebraically related to the fields we already have for each $(k,l)$. We can thus extend our polarisation relation to calculate these two extra "diagnostic" fields. Since we have now have 5 fields in 2D, modify the 2D_animate function to plot these on 5 seperate plots.

(d) Comment on whether $\vec c_p=\vec c_g$ analytically, for these waves. Does this still mean, as in 1D, that the whole system is nondispersive (the waveform never evolves if we have a packet of waves *of the same family*), or does the 2D setting allow a form of dispersion regardless?